In [ ]:
from matplotlib import font_manager
import matplotlib.pyplot as plt

font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
print("使用字体:", font_name)


# 🧱 LangChat 心智模型 | Week13-Day3
# 📌 Retail Analytics：排队分析/货架陈列/缺货检测 —— 零售场景的 Vision 能力如何转化为 KPI？

> 术语说明：LangChat 已按 ADR-008 更名为 LnkChat，本文按 W12-D7 决议使用新名（首现标注），历史引用保持原文。
> 现状声明：域知识.md「不做什么」白纸黑字写着**「客流统计 / 访问者分析（不是客流分析系统）」**。所以今天是**蓝图推演**——不是考古现有代码，而是回答：如果做零售场景，转化链路上每一环缺什么。这与 W13「能力蓝图」的定位一致。

━━━ 1. 今日核心问题 ━━━

**零售场景的 Vision 能力如何转化为 KPI？**

前两天的痛点都是"检测质量"：客流统计偏差 20%（Day1），安全告警误报率高（Day2）。今天换一个性质完全不同的问题：零售场景的检测其实**不难**（数排队人头、看货架空没空，YOLO 级能力），难的是**转化**——


In [ ]:
像素 → 检测框 → 事件/状态 → 指标(Metric) → KPI → 决策


这条"语义装配线"上，检测框之后的每一步都不是视觉问题，是**测量问题**：排队人数怎么变成"平均等待时间"？抽样式截图对连续指标意味着什么？检测偏差怎么污染月度 KPI？**安全场景卖的是"告警"，零售场景卖的是"报表"——产品形态从事件驱动变成统计驱动，领域模型、数据存储、售卖方式全都要换**。这就是今天的问题：转化的瓶颈不在眼睛，在度量衡。

━━━ 2. 人话解释（用 26 年 ERP 经验讲）━━━

Jason，你在 ERP 里做了 26 年的这件事，就是今天的事：**把业务事实变成管理层认账的数字**。

你想想月结：仓库台账 → 出入库单据 → 加权平均成本 → 毛利报表。财务从来不会说"系统抓到的单据就是毛利"——中间有一整套**口径**：哪些单算当月、退货怎么冲、赠品算不算销量。审计认的是口径，不是抓取。**零售视觉 KPI 完全同构：检测框是"单据"，KPI 是"毛利"，中间缺的那层叫指标口径（Metric Contract）。**

三个 ERP 人秒懂的点：

**① 检测偏差 ≠ 检测噪声，这是"统计型系统"和"事件型系统"的分水岭。**
昨天的教训是：安全场景一次误报都在消耗信任本金。零售场景反过来——单次检测错一点**根本不要紧**，一天 1440 帧采样平均下来，随机噪声自己消失（大数定律）。要命的是**系统性偏差**：收银台前一个人挡住后面三个人，你永远少数 3 个，平均一万次还是少数 3 个。噪声是随机误差，平均可消；偏差是口径错误，**平均只会把它固化**。ERP 的对应物：某仓管永远漏录一张领料单，月均成本看起来很稳定——稳定地错。

**② 有些指标"测不到"，但能"算出来"——Little's Law 就是你的加权平均公式。**
排队**长度**（画面里几个人）每帧可见，不需要 ID；排队**等待时间**（一个人站了多久）按 Day1 的谱系需要 ID 才能测。但排队论有个百年老定理：**L = λ × W**（平均队长 = 到达率 × 平均等待）。于是 W = L / λ——**用两个不需要 ID 的可测量（队长 L、到达率 λ），把需要 ID 的指标（等待 W）算出来**。这就是"转化"的精髓：不是把检测做深，是把测量换成估计。你做成本核算时不盘点每颗螺丝，用 BOM × 产量算出来，是同一个思维。

**③ KPI 不改变决策就不是 KPI，是壁纸。**
"周末餐饮层平均排队 11 分钟"本身不值钱；值钱的是它触发"周六 11:30-13:00 加开 2 个收银台"。POS 数据只能告诉你成交了多少，**答不了成交前的漏斗**：等太久走了多少人（弃单率）、货架缺货错失多少销售（OSA 缺货率）、陈列不合规影响多少品牌督导。视觉 KPI 的定价权就在 POS 回答不了的那段。

━━━ 3. LangChat（LnkChat）架构位置 ━━━

五层模型里，零售场景的重心和前两天都不一样——**重心第一次落在 L4**：


In [ ]:
L1  Image Understanding     — person 检测/货架目标/空位            ← yolo_world 可达（未启用）
L2  Video Understanding     — 队列跟踪（可选！见 Little's Law）    ← 非必需，降级可绕
L3  Scene Understanding     — 队列 ROI 占用/货架分区/陈列比对      ← 规则引擎部分同构
L4  Business Intelligence   — 等待时间/服务速率/OSA/陈列合规 KPI   ← 【今日重心】完全缺失
L5  Vision Agent            — 自动解读 KPI + 建议 + 日报           ← 明天主题


三个零售场景的分层落点：

| 场景 | 最小可行层 | 关键 KPI | 现状 |
|---|---|---|---|
| 排队分析 | L1（数人头）+ L4（算 W） | 平均/峰值等待、弃单率 | ❌ 无 person 闭集检测器，无 Metric 层 |
| 货架陈列 | L1 + L3（与 planogram 比对） | 陈列合规率 | ❌ floor_cleanliness 思路可迁移（基线比对） |
| 缺货检测 | L1 + L4（时序聚合） | OSA 在架率 | ❌ yolo_world 词表可达，无统计层 |

结构判断：**安全场景逼你上 L2（时序证据对抗误报），零售场景逼你上 L4（统计聚合产生 KPI）**。两条升级路线花的钱不在同一个地方——前者买检测与时序，后者买指标工程。MallSenseAI 现有资产（ROI/规则/告警）对安全路线是半成品，对零售路线是**三分之一成品**：有前半段（检测→规则），完全没后半段（指标→KPI→报表）。

━━━ 4. ADR / 战略文档依据 ━━━

**① 域知识.md「不做什么」——最诚实的边界声明**：客流统计被明确排除，理由是定位（"不是客流分析系统"）。注意这个排除的**合理性在瓦解**：排除时产品是"安防巡检"，现在 W13 在讨论它成为 LangChat 行业能力包——能力包的边界由 capability 划分，不由产品名划分。**这条边界是当年产品定位的产物，W13-D5 讨论 LangChat 集成时它必须重新裁决**（记录为 ADR Health Check 候选项）。

**② 域知识.md 设计决策 #2（封闭系统有意为之）的反面证据**：封闭的理由是"物业安防场景的实时性要求高（秒级告警），经编排网关中转会增加延迟"。把这条逻辑反过来读：**凡是容忍分钟级延迟的场景，就没有理由封闭**。零售 KPI 恰好是分钟级甚至日级容忍的统计量——**按产品自己的架构逻辑推演，Retail Analytics 应该是 MallSenseAI 打开封闭系统、暴露第一个 capability 的最佳切口**（`retail.kpi.query`，与 P1 候选 `safety.alert.query` 相比无实时性硬约束）。这是一个用现有 ADR 论证新方向的范例：不用新证据，只用旧决策的边界条件。

**③ 域知识.md「AI 产品族架构」的层次责任**：MallSenseAI = 物理感知层，LnkChat = 数据世界（知识/编排/工作流）。KPI 转化链路的自然分工由此得出：**像素→检测→事件归感知层，指标→KPI→报告归数据世界层**。今天推演的 Metric 层放在哪一层，就是这个架构决策的具体化——留作思考题。

**④ 需求证据表的边界登记方式**：客流统计被登记为 `explicitly-not-do, high, 产品定位`——不是"没钱做"，是"定位不做"。**ERP 经验：定位不做的清单比待办清单更值得定期复审**，因为定位会漂移（现在就在漂）。

━━━ 5. 代码验证（只看关键结构）━━━

**现有资产离零售 KPI 有多远？逐个检查关键结构：**

**① 检测契约是单帧无状态的——对零售 KPI 反而是好消息**（`backend/app/detectors/base.py`）：


In [ ]:
@dataclass(frozen=True)
class DetectionResult:
    polygon: list[Point]     # 归一化坐标
    confidence: float
    label: str
    metadata: dict[str, Any]   # ← 任意附加数据，指标装配的现成挂点

class BaseDetector(abc.ABC):
    async def detect(self, image_bytes, roi_polygons, config) -> list[DetectionResult]: ...


队列长度 = queue ROI 内 person 检测计数，**每次调用就能产出，不需要跨帧状态**——Day1 谱系里"密度/队长"恰好是无 ID 可降级段。契约完全够用。

**② 开放词表是零售的现成入口**（`yolo_world.py`）：


In [ ]:
DEFAULT_WORLD_CLASSES = ["shopping cart", "cardboard box", "stroller", ...]
classes = config.get("classes") or config.get("debris_classes")
self._apply_classes(model, classes)      # 运行时 set_classes，无需重训


把 prompt 换成 `"person"` / `"empty shelf"` 就是零售检测器——**检测层零新增代码，纯配置**。OpenSpec（obstruction-congestion spec）甚至已定义了"操作员增删检测词表"的场景。

**③ 抽样节奏已存在**（`workers/scheduler.py`）：


In [ ]:
def default_interval_seconds(self) -> float:
    return max(1.0, float(self._settings.alarm_interval_minutes * 60))


每相机分钟级巡检 = 对连续量的周期抽样。Day1 结论说"累积型指标误差是噪声级"——零售 KPI 恰是累积型。**抽样基础设施已经在了，只是从来没人为"指标置信区间"这个目的用它**。

**④ 缺的那块：全仓库没有 Metric/TimeSeries 实体**。`models/entities.py` 的领域对象全部是事件型（Alert 及其状态机：pending → confirmed / false_positive / resolved）。**"平均等待时间"在这套领域模型里无处安放**——它不是告警，没有生命周期状态机，它是时间序列聚合。这就是转化的第一硬缺口：**需要一个新的领域对象（MetricSample / KPIReport），这是 ADR 级决策，不是加张表**。

**⑤ 规则引擎的 dwell 机制可以迁移但语义要换**（`rules/engine.py`）：`min_stay_seconds` 判"持续占用才是事件"，本质是 ROI 占用时长——队列的"队伍持续存在"可用同构机制判，但**等待时间的语义必须靠 Little's Law 换算**（见 §2②），dwell 计的是"ROI 被占用多久"而不是"每个人等多久"（Day1 谱系的失真边界）。

**结论：检测层 60% 现成（配置级）、调度层 100% 现成、规则层可迁移、指标层 0%——转化链路的缺口精确落在 L4。**

━━━ 6. 商业地产映射 ━━━

| MallSenseAI/Vision 概念 | MI CRE 场景 | KPI | 触发的决策 | 责任人 |
|---|---|---|---|---|
| queue ROI + person 计数 | 餐饮层收银/服务台/电影院售票 | 平均等待分钟、峰值等待 | 高峰加开收银/引导自助点单 | 运营部 |
| 等待 × POS 流水关联 | 等待 >8min 时段 vs 成交量 | 弃单率估算 | 排班模型调整、动线改造立项 | 运营 + IT |
| 货架空位检测（超市主力店） | 租户货架在架率 OSA | OSA < 95% 预警 | 租户经营健康度前导指标 | 租户管理部 |
| 陈列与 planogram 比对 | 品牌专柜陈列合规 | 合规率 | 合同陈列条款执行督导 | 市场部 |
| KPI 时序库 + 阈值 | 各楼层分时段热/冷区 | 通过率、停留分布 | 招商定价、铺位调整、活动评估 | 招商部 |

**给 MI 的战略提醒（两条）**：

其一，**OSA 是商场对租户的"前导信用指标"**：销售数据租户可以不给，货架是自己长在卖场里的——视觉看到的就是合同谈判桌上能用的。这解释了为什么零售视觉在 CRE 的买方常常是**业主而不是商户**：业主需要看进租赁物里的经营真相。

其二，**排队 KPI 是"体验型"卖场的刚需**：机场、医院、政务服务、乐园都在用同一套技术——MI 管理的购物中心 + 长租公寓 + 写字楼门禁大堂全部适用。这是 MallSenseAI 从"商场"扩展到"资产组合"的自然外延。

━━━ 7. 与传统方案比较 ━━━

**A. 视觉排队 vs 传感器排队（红外对射/压力垫/Wi-Fi 探针）**：
- 传感器：便宜、成熟，但每点位单件安装、只测"过线/在位"，队列形状/弃单/多队合并全瞎；
- 视觉：复用存量 IPC（域知识里"摄像头改造成本"的痛点在这里反转为优势——零售场景本来就装了对着收银/货架的枪机）、一个模型服务所有点位、可回溯看画面复核；
- 视觉的代价：遮挡偏差（收银台挡人）、隐私合规（必须人头级不做人脸——域知识的人脸红线继续有效）。

**B. 视觉 KPI vs POS 数据分析**：
- POS：免费（本来就有）、精确（每一笔交易）、但只覆盖**成交后**；
- 视觉：新建成本、有估计误差、覆盖**成交前**（等待、弃单、动线、缺货错失）；
- 结论不是替代是拼图：**POS 是结果，视觉是过程**。排队 × POS 关联分析（等待高的时段 vs 客单变化）才是弃单率这类 KPI 的完整算法——单一数据源做不出这个指标。

**C. 自建 vs 采购客流数据服务（汇纳类）**：
- 采购：单店年费模式、黑盒口径、数据不在自己手里、和自有系统无集成；
- 自建（MallSenseAI 路线）：数据主权、口径可审计（MI 作为管理方可向业主解释数字怎么来的）、沉淀为 LangChat capability 可复用；
- 自建的成立条件：**多资产组合摊薄成本**——单商场自建不如买，资产组合自建才是平台生意。这正是它该长在 LangChat 能力包体系里而不是独立卖的原因。

**D. 转化路线之争：跟踪测量 vs 稳态估计（今日核心技术决策）**：
- 跟踪测量（ByteTrack 级）：每人进队/离队时间戳，W 直接可得，但需要视频流 + L2 建设（W12-D4 的架构改造全要）；
- 稳态估计（Little's Law）：W = L/λ，两个 L1 可测量换一个 L2 指标，**现有截图架构就能跑**；
- 代价：估计式 `W = L/λ` 的病态区**不在高峰在低峰**——分母 λ̂ 趋零时，除法把微小误差放大成分钟级假等待（画面里仅看一两个人，往往还是正在服务的那位）；高峰窗口样本最密集、估计反而最贴线。因此 KPI 报表的窗口粒度必须**按流量分级**（高峰 30 分钟窗，低峰 2 小时窗或直接报"无排队"），不是全局统一（今天的 notebook 实验 2/3 给出定量版：低流量窗口误差标准差高出 1.4 倍且均值漂移更大）；**架构判断：先用稳态估计上 KPI MVP，跟踪等视频流架构成熟再换——测量精度是可演进的，架构不必一步到位**。

━━━ 8. 架构师思考题 ━━━

**题 1（Metric 层归属）**：给 MallSenseAI 加指标层，三个放法：(a) backend 里加 MetricSample 表 + 定时聚合任务；(b) 独立 metrics worker 写时序库（如按天分片的聚合表）；(c) 只存原始检测事件，KPI 计算全推给 LangChat 侧（capability 返回事件流，聚合在数据世界层做）。用域知识.md 的层次责任（物理感知层 vs 数据世界层）+ 封闭系统的延迟论证，评估三个方案各自把哪条架构原则逼到墙角？你选哪个，KPI 口径变更（比如"等待"定义改了）时谁买单？

**题 2（KPI 的政治学）**：商场用视觉检测租户（超市主力店）的货架空置率并纳入租户考核。租户抗辩三条：检测不准（举证责任在谁？）、经营隐私（合同里数据共享条款怎么写？）、只看到货架看不到后仓（指标公平性怎么保证？）。设计这个 KPI 的"口径合同"：采样时段、置信区间、复核申诉流程。提示：ERP 里"盘点差异率"纳入仓管 KPI 时踩过的坑，这里一个不少。

**题 3（抽样预算）**：物业 SLA 要求"高峰时段（11:30-13:30）平均排队等待的月度 KPI 误差 ≤ ±1 分钟"。已知抽样间隔 `alarm_interval_minutes` 可调、每次抽样有 ±1 人的检测噪声、遮挡偏差 −15% 需一次性校准。推导：间隔从 10 分钟压到 1 分钟，随机误差缩小多少（√N 规律）？花在加密抽样的钱 vs 花在校准的钱，边际收益曲线长什么样？（今天的 notebook 实验 3 就是这道题的定量版。）

━━━ 9. 我的理解变化 ━━━

**以前以为**：零售视觉分析的门槛是检测技术——识别排队的人、识别空货架，模型够好产品就成立；MallSenseAI 不做零售是因为没做这些检测器。
**现在知道**：检测是零售场景里**最便宜的一环**（yolo_world 换个 prompt 就上线），贵的和难的全在检测之后：指标口径（W 怎么定义）、测量方法（Little's Law 用可测换不可测）、抽样方案（√N 与校准）、KPI 合同（口径、置信度、申诉）。**MallSenseAI 不做零售的真正原因不是没检测器，是整个产品形态是事件驱动（Alert/工单），零售需要统计驱动（Metric/报表）——差一个领域模型，不是差一个模型。**

**第二个变化**：以前把"误报率"当成视觉系统唯一的质量敌人（昨天 Day2 的视角）。现在知道**不同场景的错误经济学完全不同**：安全的误报是信任死刑（离散事件，一次都多）；零售的偏差是慢性毒（连续统计，单次无所谓，但永远偏 15% 就毁掉 KPI）。治理手段随之分岔：安全场景要多层漏斗压制单次误报，零售场景要一次性校准 + 大数定律。**同一套检测基础设施，在不同场景要配完全不同的"质量工程"——这是把 MallSenseAI 做成能力包时必须显式建模的东西（capability 的元数据里应该带场景级质量策略）。**

━━━ 10. 明日连接 + Semantic Layer ━━━

**明天（Day 4）**：Vision Agent——从"检测到一个人"到"自动分析→推理→建议→日报"。今天造的 KPI 恰是 Vision Agent 的口粮：没有 L4 的指标，L5 的 Agent 只能看图说话（VLM 描述单帧），有了指标才能做趋势归因和行动建议。**今天的问题是"怎么把像素变成数字"，明天的问题是"数字谁来看懂"**——以及 LnkChat Agent 和 Vision Agent 的边界到底画在哪（同一套 governance 能不能复用）。

**Semantic Layer 位置**：


In [ ]:
Ontology（零售域：队列/服务/货架/在架率/弃单 —— "度量衡"本体）
  → Domain Model（候选新对象：MetricSample, KPIDefinition, CalibrationRecord
                  —— 与 Alert 并列的统计型对象，ADR 级新增）
    → Capability（retail.kpi.query / retail.queue.report —— 打破封闭的首选切口：
                  分钟级容忍天然适配 LangChat 编排，不撞安全场景的秒级红线）
      → Skill（运营日报数字员工：晨会前自动生成"昨日排队/缺货/陈列三表 + 建议"）


今天的核心收获在链条的最左端：**Ontology 层的零售域不是"商品/店铺"，是"度量衡"——先定义什么是等待、什么算缺货，后面的 Domain Model 才知道 Metric 长什么样**。这和 LnkChat 的 Domain Model 先于 Capability 是同一条纪律：**对象没定义清楚之前，能力只是脚本**。
